# 05. Object Localization
___

Pipeline:

Image

  ↓

Grounding DINO

  ↓

Product bounding box

  ↓

SAM/SAM2

  ↓
  
Product segmentation mask

Mục tiêu:

- localize product
- obtain bounding box
- obtain segmentation mask
- calculate object area ratio
- save localization metadata

Không xóa background ở bước này.
Chỉ tạo mask/bbox để các experiment sau sử dụng.

## 1. Configuration

In [ ]:
!pip install -q transformers accelerate torch torchvision

In [ ]:
GROUNDING_DINO_MODEL = (
    "IDEA-Research/grounding-dino-base"
)

SAM_MODEL = (
    "facebook/sam-vit-base"
)

TEXT_PROMPT = "product."
BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25

## 2. Grounding DINO

Pseudo-implementation theo Transformers:

In [ ]:
import torch
from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

processor = AutoProcessor.from_pretrained(
    GROUNDING_DINO_MODEL
)

model = (
    AutoModelForZeroShotObjectDetection
    .from_pretrained(
        GROUNDING_DINO_MODEL
    )
    .to(device)
)

## 3. Detection

In [ ]:
def detect_product(image,prompt="product."):
    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    result = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        target_sizes=[
            image.size[::-1]
        ]
    )[0]
    return result

## Visualize bounding box

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def visualize_boxes(image,result):
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    for box, score in zip(
        result["boxes"],
        result["scores"]
    ):
        x1, y1, x2, y2 = (
            box.cpu().numpy()
        )
        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            y1,
            f"{score:.2f}"
        )
    ax.axis("off")
    plt.show()